# Ising Model: Autocorrelation and Equilibration Analysis

This notebook compares three update schemes for the 2D Ising model from the same deliberately unfavorable starting point: a random spin configuration with no warm-up. Tracking the early-time magnetization makes the contrast between local Metropolis updates and the Wolff cluster move visually obvious, especially near the Onsager critical temperature $T_c = 2/\ln(1 + \sqrt{2}) \approx 2.269$ [[1]](#Bibliography).

The goal is not a precision measurement of autocorrelation times. Instead, the notebook shows how quickly each algorithm settles into its typical magnetization range below, at, and above $T_c$, and it stores the generated trajectories in a small notebook cache so repeated reads do not rerun the same simulation every time [[2]](#Bibliography) [[3]](#Bibliography).

In [ ]:
from __future__ import annotations

import os

import matplotlib.pyplot as plt
import numpy as np

from models.ising_model import IsingSimulation

TC_ISING = 2.0 / np.log(1.0 + np.sqrt(2.0))
L = 64
N_STEPS = 1000
TEMPS = [0.8 * TC_ISING, TC_ISING, 1.2 * TC_ISING]
TEMP_LABELS = ['Below $T_c$', 'At $T_c$', 'Above $T_c$']
UPDATES = ['random', 'checkerboard', 'wolff']
UPDATE_LABELS = {
    'random': 'Random-site Metropolis',
    'checkerboard': 'Checkerboard Metropolis',
    'wolff': 'Wolff cluster',
}
PALETTE = {'random': '#4878CF', 'checkerboard': '#6ACC65', 'wolff': '#D65F5F'}
CACHE_PATH = '../results/ising/ising_autocorrelation_analysis.npz'

print(f'Exact Onsager T_c = {TC_ISING:.6f}')
print(f'Notebook cache path: {CACHE_PATH}')

## Load or compute trajectories

If a cached trajectory file exists in `../results/ising/ising_autocorrelation_analysis.npz`, the notebook reuses it directly. Otherwise it runs the three temperature slices inline and saves the resulting $|M|$ time series for later reads. This keeps the notebook self-contained even though there is no separate batch script for this specific figure yet.

In [ ]:
if os.path.exists(CACHE_PATH):
    data = np.load(CACHE_PATH, allow_pickle=False)
    magnetization_traces = data['magnetization_traces']
    print(f'Loaded cached trajectories from {CACHE_PATH}: shape={magnetization_traces.shape}.')

else:
    print('Cached trajectories not found. Running inline simulations and saving the result...')
    magnetization_traces = np.empty((len(TEMPS), len(UPDATES), N_STEPS), dtype=float)

    for i, temperature in enumerate(TEMPS):
        for j, update in enumerate(UPDATES):
            sim = IsingSimulation(size=L, temp=temperature, update=update, seed=42)
            mags, _ = sim.run(n_steps=N_STEPS)
            magnetization_traces[i, j] = np.asarray(mags, dtype=float)
            print(
                f'  T={temperature:.3f}, update={update}: recorded {N_STEPS:,} steps',
                flush=True,
            )

    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    np.savez(
        CACHE_PATH,
        temperatures=np.asarray(TEMPS, dtype=float),
        temp_labels=np.asarray(TEMP_LABELS),
        updates=np.asarray(UPDATES),
        magnetization_traces=magnetization_traces,
    )
    print(f'Saved cached trajectories to {CACHE_PATH}.')

## Magnetization traces

Each panel shows the absolute magnetization $|M|$ over the first 1,000 updates after a random start. The comparison is deliberately qualitative: below $T_c$ the system should lock into an ordered state, near $T_c$ the trajectories should remain noisy for much longer, and above $T_c$ they should fluctuate around a small mean.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharex=True, sharey='row')
fig.suptitle(
    f'Absolute magnetization from random starts over {N_STEPS:,} steps (L = {L})',
    fontsize=14,
    y=1.02,
 )

for i, temp_label in enumerate(TEMP_LABELS):
    for j, update in enumerate(UPDATES):
        ax = axes[i, j]
        ax.plot(
            np.abs(magnetization_traces[i, j]),
            lw=1.0,
            color=PALETTE[update],
        )
        if i == 0:
            ax.set_title(UPDATE_LABELS[update], fontsize=11)
        if j == 0:
            ax.set_ylabel(f'{temp_label}\n$|M|$')
        if i == len(TEMPS) - 1:
            ax.set_xlabel('Monte Carlo step')
        ax.grid(alpha=0.25)

fig.tight_layout()
plt.show()

## What changes across temperature

Below $T_c$, all three algorithms drive the system toward large $|M|$, but the Wolff traces typically reach that regime fastest because a single cluster flip can reorganize an extended domain. At $T_c$, the contrast is sharpest: both local Metropolis variants spend much longer wandering through correlated configurations, while Wolff decorrelates more aggressively. Above $T_c$, the three methods look more similar because the correlation length is short and there is less structure for a cluster move to exploit [[4]](#Bibliography).

## Bibliography

[[1]](#Bibliography) L. Onsager, "Crystal Statistics. I. A Two-Dimensional Model with an Order-Disorder Transition," *Physical Review*, vol. 65, no. 3-4, pp. 117–149, 1944. [APS Open Access](https://journals.aps.org/pr/abstract/10.1103/PhysRev.65.117)

[[2]](#Bibliography) W. K. Hastings, "Monte Carlo sampling methods using Markov chains and their applications," *Biometrika*, vol. 57, no. 1, pp. 97–109, 1970. [Oxford Academic Open Access](https://academic.oup.com/biomet/article/57/1/97/252073)

[[3]](#Bibliography) U. Wolff, "Collective Monte Carlo Updating for Spin Systems," *Physical Review Letters*, vol. 62, no. 4, pp. 361–364, 1989. [APS Open Access](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.62.361)

[[4]](#Bibliography) M. E. J. Newman and G. T. Barkema, "Monte Carlo Methods in Statistical Physics," Oxford University Press, 1999. [Lecture Notes Summary (H. G. Katzgraber)](https://arxiv.org/abs/0905.1629)